# 02 · Централизованный поиск похожей музыки
Запускайте ячейки сверху вниз. Один запрос — это одна любимая композиция. Сервис сравнивает её с каталогом на одном узле и возвращает пять других ID. В конце можно изменить название и проверить свою песню.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


### 1. Загружаем результат очистки
В tracks.csv каждый ID уникален. Жанры добавляем для чтения результатов. Они не участвуют в расчёте сходства.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, normalize
from time import perf_counter

folder = Path("/content/drive/MyDrive/Colab Notebooks/Spotify_ARD/data/processed")
tracks = pd.read_csv(folder / "tracks.csv").sort_values("track_id").set_index("track_id")
links = pd.read_csv(folder / "track_genres.csv")
assert tracks.index.is_unique and set(links.track_id) == set(tracks.index)
tracks["genres"] = links.groupby("track_id").track_genre.agg(", ".join)
print("Треков:", len(tracks), "| жанров:", links.track_genre.nunique())

Треков: 89740 | жанров: 114


### 2. Представляем звучание числами
Семь признаков уже лежат в диапазоне 0–1. Темп и громкость приводим к тому же диапазону по каталогу. Нулевой темп заменяем медианой положительных значений как простое приближение. Оно сохраняет запись в выдаче, но может сделать трек ближе к композициям среднего темпа.

Нормируем каждую строку до единичной длины. Тогда скалярное произведение равно косинусному сходству. Матрицу всех попарных расстояний не строим.

In [3]:
features = ["danceability", "energy", "speechiness", "acousticness",
            "instrumentalness", "liveness", "valence", "tempo", "loudness"]
X = tracks[features].copy()
known_tempo = X.loc[X.tempo > 0, "tempo"]
print("Замен нулевого темпа:", X.tempo.eq(0).sum(), "| медиана BPM:", known_tempo.median())
X.loc[X.tempo.eq(0), "tempo"] = known_tempo.median()
scaler = MinMaxScaler()
X[["tempo", "loudness"]] = scaler.fit_transform(X[["tempo", "loudness"]])
assert np.isfinite(X).all().all() and (X.abs().sum(axis=1) > 0).all()
matrix = normalize(X.to_numpy(dtype="float64"))

Замен нулевого темпа: 157 | медиана BPM: 122.021


### 3. Полный перебор и объяснение результата
Исключаем также другие ID с тем же исполнителем и названием. Из совпадающих по этой паре кандидатов оставляем лучший. Такая эвристика может объединить разные записи с одинаковой подписью. Сортируем по убыванию сходства. При равных оценках сохраняется порядок ID. В пояснении указываем два признака с наименьшей абсолютной разницей после масштабирования. Это описание пары, а не разложение косинусной оценки.

In [4]:
song_keys = list(zip(tracks.artists.str.strip().str.casefold(), tracks.track_name.str.strip().str.casefold()))

def recommend(track_id, k=5):
    position = tracks.index.get_loc(track_id)
    scores = matrix @ matrix[position]
    scores[position] = -np.inf
    seen = {song_keys[position]}
    order = []
    for i in np.argsort(-scores, kind="stable"):
        if song_keys[i] in seen:
            continue
        seen.add(song_keys[i])
        order.append(i)
        if len(order) == k:
            break
    result = tracks.iloc[order][["track_name", "artists", "genres"]].copy()
    result["сходство"] = scores[order].round(5)
    differences = X.iloc[order].sub(X.iloc[position]).abs()
    result["близкие признаки"] = [", ".join(row.nsmallest(2).index)
                                   for _, row in differences.iterrows()]
    return result.reset_index()

### 4. Выберите трек
Измените название ниже и выполните ячейку. Если в каталоге несколько версий, таблица покажет варианты. Для точного выбора подставьте нужный ID в recommend. По умолчанию берём первый ID из найденных.

In [5]:
title = "Mr. Brightside"
matches = tracks.loc[tracks.track_name.str.casefold().eq(title.casefold()),
                     ["track_name", "artists", "genres"]]
if matches.empty:
    print("Точного совпадения нет. Попробуйте другое название.")
else:
    display(matches.head(10).reset_index())
    seed_id = matches.index[0]
    print("Выбранный ID:", seed_id)
    display(recommend(seed_id))

,track_id,track_name,artists,genres
0,003vvx7Niy0yvhvHt4a68B,Mr. Brightside,The Killers,"alt-rock, alternative, rock"
1,23PvWFdi76vER4p1e2Xroj,Mr. Brightside,The Killers,"alt-rock, alternative, dance, rock"
2,315FrAki9HoxEdG8CQuZhD,Mr. Brightside,The Killers,"alt-rock, alternative, dance, rock"
3,52cFhaPuS5oLrmFoJw4J9j,Mr. Brightside,The Killers,"alt-rock, alternative, dance, rock"
4,5KaVoEhxf1UwParyAjzHVA,Mr. Brightside,The Killers,"alt-rock, alternative, dance, rock"


Выбранный ID: 003vvx7Niy0yvhvHt4a68B


,track_id,track_name,artists,genres,сходство,близкие признаки
0,7bAPKF3x8O8g6NVhDfeVhM,Iron Cage,Toxic Holocaust,"black-metal, death-metal",0.99944,"acousticness, liveness"
1,2VSbEXqs6NbNiZSTcHlIDR,My Sacrifice,Creed,"alt-rock, grunge, metal",0.99939,"instrumentalness, acousticness"
2,54otdw8MmYNz2ewGHzmaTP,FEEL NOTHING - RESET,The Plot In You,"metal, metalcore",0.99929,"instrumentalness, acousticness"
3,46cEIa1iQPSQLnPVP2hmVm,Jesse Intense,The Amity Affliction,metalcore,0.99928,"instrumentalness, acousticness"
4,3I8w8mVY4KFBXbhcP5NqqH,For Victory,Burgerkill,death-metal,0.99916,"instrumentalness, acousticness"


### 5. Проверяем выдачу и измеряем задержку
Берём 20 разных ID с фиксированным случайным зерном. Для каждого проверяем размер, уникальность и исключение запроса. Измерение охватывает поиск и подготовку таблицы после загрузки каталога. Медиана описывает типичный запрос, 95-й процентиль — верхнюю часть наблюдаемых задержек. В Colab числа могут отличаться.

In [6]:
queries = tracks.sample(20, random_state=42).index
recommend(queries[0])  # прогрев
elapsed = []
for track_id in queries:
    start = perf_counter()
    result = recommend(track_id)
    elapsed.append((perf_counter() - start) * 1000)
    assert len(result) == result.track_id.nunique() == 5
    assert track_id not in result.track_id.values
    keys = {song_keys[tracks.index.get_loc(i)] for i in result.track_id}
    assert len(keys) == 5 and song_keys[tracks.index.get_loc(track_id)] not in keys
    assert result["сходство"].is_monotonic_decreasing
print("Проверено запросов:", len(queries))
print(f"Медиана: {np.median(elapsed):.1f} мс | p95: {np.percentile(elapsed, 95):.1f} мс")

Проверено запросов: 20
Медиана: 22.5 мс | p95: 25.5 мс


### Вывод
Получена воспроизводимая исходная выдача для сравнения следующих методов. Проверены технические свойства топ-5. Чтобы оценить попадание во вкус слушателя, понадобятся его ответы на предложенные пары.

## Обучение на моделируемых предпочтениях
Следующие разделы дополняют поиск по косинусному сходству экспериментом с обучением. Все ответы слушателей синтетические. Они генерируются из реальных аудиопризнаков и жанров Spotify. Результаты характеризуют выбранную симуляцию. Перенос на живых слушателей потребует отдельной проверки.

Три повторения с разными зернами случайности показывают чувствительность к разделению каталога и генерации ответов. Нельзя интерпретировать разброс трёх запусков как доверительный интервал для реальных пользователей.

In [2]:
from pathlib import Path
import json, hashlib, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from time import perf_counter
from scipy.special import expit
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, log_loss
import sklearn

root = Path("/content/drive/MyDrive/Colab Notebooks/Spotify_ARD")
out = root / "experiment"
out.mkdir(exist_ok=True)
# Можно открыть блокнот 03 сразу после получения архива результатов.
import zipfile
archive = root / "experiment_results.zip"
if archive.exists() and not (out / "simulation_42.npz").exists():
    with zipfile.ZipFile(archive) as bundle:
        for item in bundle.infolist():
            if Path(item.filename).name != item.filename:
                raise ValueError("Ожидался архив с плоской структурой")
            (out / item.filename).write_bytes(bundle.read(item))
SEEDS = [42, 123, 2026]


### Каталог и разделение до преобразований
Объединяем одинаковые подписи исполнителя и названия до разделения. Затем случайно делим кандидатов на train/validation/test в пропорции 60/20/20. Одинаковые подписи не пересекают границы частей. Профиль слушателя задаёт один любимый трек из train, общий для его запросов. MinMaxScaler и медиана темпа рассчитываются только по train. Значения вне обучающего диапазона сохраняются.

In [4]:
audio = ["danceability", "energy", "speechiness", "acousticness",
         "instrumentalness", "liveness", "valence", "tempo", "loudness"]
catalog = pd.read_csv(root / "data/processed/tracks.csv").sort_values("track_id")
links = pd.read_csv(root / "data/processed/track_genres.csv")
catalog["song_key"] = (catalog.artists.str.strip().str.casefold() + "\0" +
                       catalog.track_name.str.strip().str.casefold())
catalog = catalog.drop_duplicates("song_key").reset_index(drop=True)
genre_map = links.groupby("track_id").track_genre.agg(set)
print("Уникальных подписей исполнителя и названия:", len(catalog))


Уникальных подписей исполнителя и названия: 81206


### Генератор трёх слушателей
У каждого слушателя собственные веса различий звучания, порог принятия и любимый трек. Вероятность положительного ответа зависит от расстояния до любимой песни, пересечения жанров и нелинейного взаимодействия энергии с эмоциональностью. Затем ответ случайно выбирается из распределения Бернулли. Даже похожая песня может получить отрицательную оценку.

Генератор намеренно использует заданные исследователем зависимости. Модель учится воспроизводить их по примерам. Это не независимое доказательство качества музыкальных рекомендаций. Для каждого клиента выбираем 2000/500/500 разных кандидатов из соответствующих частей. Половина — близкие по признакам, половина — из оставшегося пула. Клиентские размеры одинаковы, вкусы и доли положительных ответов различаются.

Модель получает различия девяти признаков и пересечение жанров, а также их взаимодействия с идентификатором одного из трёх известных слушателей. Это позволяет одной глобальной модели представлять три вкуса. Обобщение на нового слушателя здесь не проверяется.

In [6]:
def simulate(seed):
    rng = np.random.default_rng(seed)
    permutation = rng.permutation(len(catalog))
    a, b = int(len(catalog)*0.6), int(len(catalog)*0.8)
    parts = [permutation[:a], permutation[a:b], permutation[b:]]
    assert not (set(parts[0]) & set(parts[1]) or set(parts[0]) & set(parts[2]))
    assert not set(parts[1]) & set(parts[2])
    raw = catalog[audio].copy()
    median = raw.iloc[parts[0]].loc[lambda t: t.tempo > 0, "tempo"].median()
    raw.loc[raw.tempo.eq(0), "tempo"] = median
    scaler = MinMaxScaler().fit(raw.iloc[parts[0]][["tempo", "loudness"]])
    raw[["tempo", "loudness"]] = scaler.transform(raw[["tempo", "loudness"]])
    vectors = raw.to_numpy()
    favorites = rng.choice(parts[0], 3, replace=False)
    arrays, records = {}, []
    taste = np.array([[1,3,1,1,1,1,2,2,1],
                      [2,1,1,3,3,1,1,1,1],
                      [3,1,2,1,1,2,3,1,1]], dtype=float)
    for split, pool, count in zip(["train","val","test"], parts, [2000,500,500]):
        xx, yy, cc, ids = [], [], [], []
        for client, favorite in enumerate(favorites):
            candidates = np.setdiff1d(pool, favorites)
            distances = np.abs(vectors[candidates] - vectors[favorite]).mean(axis=1)
            near = candidates[np.argsort(distances)[:max(count, len(candidates)//5)]]
            chosen_near = rng.choice(near, count//2, replace=False)
            chosen_other = rng.choice(np.setdiff1d(candidates, chosen_near), count-count//2, replace=False)
            chosen = np.concatenate([chosen_near, chosen_other])
            delta = np.abs(vectors[chosen] - vectors[favorite])
            g = genre_map[catalog.iloc[favorite].track_id]
            overlap = np.array([len(g & genre_map[track_id])/len(g | genre_map[track_id])
                                for track_id in catalog.iloc[chosen].track_id])
            score = ([1.6, 1.1, 2.0][client] - 0.8*(delta @ taste[client]) +
                     1.4*overlap + 1.5*np.sin(4*delta[:,1])*delta[:,6])
            p = expit(score)
            y = rng.binomial(1, p)
            base = np.column_stack([delta, overlap])
            onehot = np.eye(3)[np.full(count, client)]
            interaction = (base[:,:,None]*onehot[:,None,:]).reshape(count,-1)
            X = np.column_stack([np.ones(count), base, onehot, interaction])
            xx.append(X); yy.append(y); cc.append(np.full(count,client)); ids.extend(chosen)
            records.extend([{"seed":seed,"split":split,"client":client,
                "seed_id":catalog.iloc[favorite].track_id,"candidate_id":catalog.iloc[j].track_id,
                "liked":int(label),"synthetic_probability":float(prob)}
                for j,label,prob in zip(chosen,y,p)])
        arrays["X_"+split]=np.vstack(xx)
        arrays["y_"+split]=np.concatenate(yy)
        arrays["client_"+split]=np.concatenate(cc)
        arrays["track_"+split]=np.array(ids)
    for left,right in [("train","val"),("train","test"),("val","test")]:
        assert not set(arrays["track_"+left]) & set(arrays["track_"+right])
    arrays["favorite_ids"]=catalog.iloc[favorites].track_id.to_numpy(dtype=str)
    arrays["tempo_median"]=np.array(median)
    arrays["scale"]=scaler.scale_
    arrays["min"]=scaler.min_
    return arrays, pd.DataFrame(records)

all_ratings = []
for seed in SEEDS:
    arrays, ratings = simulate(seed)
    np.savez_compressed(out / f"simulation_{seed}.npz", **arrays)
    all_ratings.append(ratings)
ratings = pd.concat(all_ratings, ignore_index=True)
ratings.to_csv(out / "synthetic_ratings.csv", index=False)
display(ratings.groupby(["seed","split","client"]).liked.agg(["count","mean"]).round(3))


count   mean
seed split client              
42   test  0         500  0.534
           1         500  0.242
           2         500  0.178
     train 0        2000  0.541
           1        2000  0.248
           2        2000  0.165
     val   0         500  0.584
           1         500  0.222
           2         500  0.178
123  test  0         500  0.296
           1         500  0.218
           2         500  0.532
     train 0        2000  0.297
           1        2000  0.230
           2        2000  0.515
     val   0         500  0.302
           1         500  0.202
           2         500  0.484
2026 test  0         500  0.518
           1         500  0.164
           2         500  0.518
     train 0        2000  0.543
           1        2000  0.148
           2        2000  0.501
     val   0         500  0.516
           1         500  0.136
           2         500  0.496

### Централизованная логистическая модель
Минимизируем бинарную кросс-энтропию с L2-регуляризацией 0,001. Начальные веса нулевые. Один запуск — 500 полных градиентных шагов, контроль validation каждые пять шагов. Из скоростей 0,1/0,3/1,0 выбираем лучшую по validation log loss и возвращаем лучший промежуточный снимок. Test используется после выбора. Accuracy и F1 используют порог 0,5. ROC AUC оценивает порядок вероятностей.

Базовый прогноз — доля положительных ответов конкретного клиента в train. Он учитывает различия в склонности слушателей ставить положительные оценки.

In [8]:
def probabilities(X, w):
    return expit(X @ w)

def gradient_step(X, y, w, lr):
    grad = X.T @ (probabilities(X, w) - y) / len(y)
    penalty = 0.001 * w.copy()
    penalty[0] = 0
    return w - lr * (grad + penalty)

def fit_model(X, y, V, vy, lr, clients=None, local_steps=5, rounds=100):
    w = np.zeros(X.shape[1])
    best_w = w.copy()
    best_loss = np.inf
    history = []
    start = perf_counter()
    for round_index in range(rounds):
        if clients is None:
            for _ in range(local_steps):
                w = gradient_step(X, y, w, lr)
        else:
            updates = []
            for ids in clients:
                local_w = w.copy()
                for _ in range(local_steps):
                    local_w = gradient_step(X[ids], y[ids], local_w, lr)
                updates.append(local_w)
            w = np.average(updates, axis=0, weights=[len(ids) for ids in clients])
        loss = log_loss(vy, probabilities(V, w), labels=[0, 1])
        history.append(loss)
        if loss < best_loss:
            best_loss, best_w = loss, w.copy()
    return best_w, best_loss, np.array(history), perf_counter() - start

def metrics(y, p):
    return {"accuracy": accuracy_score(y, p >= 0.5),
            "f1": f1_score(y, p >= 0.5, zero_division=0),
            "roc_auc": roc_auc_score(y, p),
            "log_loss": log_loss(y, p, labels=[0, 1])}

def select_model(X, y, V, vy, clients=None, local_steps=5):
    candidates = []
    elapsed = 0
    for lr in [0.1, 0.3, 1.0]:
        w, loss, curve, seconds = fit_model(X, y, V, vy, lr, clients, local_steps)
        elapsed += seconds
        candidates.append((loss, lr, w, curve, seconds))
    best = min(candidates, key=lambda item: item[0])
    return best, elapsed


In [9]:
central_rows = []
central_weights = {}
for seed in SEEDS:
    d = np.load(out / f"simulation_{seed}.npz")
    best, search_seconds = select_model(d["X_train"],d["y_train"],d["X_val"],d["y_val"])
    loss, lr, w, curve, seconds = best
    prior = np.array([d["y_train"][d["client_train"]==c].mean() for c in range(3)])
    for method,p in [("Client prior",prior[d["client_test"]]),
                     ("Central",probabilities(d["X_test"],w))]:
        central_rows.append({"seed":seed,"method":method,**metrics(d["y_test"],p),
                            "validation_loss":loss if method=="Central" else np.nan,
                            "lr":lr if method=="Central" else np.nan,
                            "fit_seconds":seconds if method=="Central" else 0,
                            "search_seconds":search_seconds if method=="Central" else 0})
    np.savez(out / f"central_{seed}.npz", weights=w, curve=curve, lr=lr)
central_results = pd.DataFrame(central_rows)
central_results.to_csv(out / "central_results.csv", index=False)
display(central_results.round(4))
display(central_results.groupby("method")[["accuracy","f1","roc_auc","log_loss"]].agg(["mean","std"]).round(4))


,seed,method,accuracy,f1,roc_auc,log_loss,validation_loss,lr,fit_seconds,search_seconds
0,42,Client prior,0.7047,0.5466,0.6824,0.5711,NaN,NaN,0.0000,0.0000
1,42,Central,0.7527,0.5691,0.7742,0.5193,0.5094,1.0,0.1615,0.4894
2,123,Client prior,0.6727,0.5200,0.6536,0.6080,NaN,NaN,0.0000,0.0000
3,123,Central,0.7327,0.5853,0.7966,0.5126,0.5048,1.0,0.1462,0.4721
4,2026,Client prior,0.6240,0.6475,0.6639,0.6114,NaN,NaN,0.0000,0.0000
5,2026,Central,0.6907,0.6548,0.7617,0.5391,0.5271,1.0,0.1404,0.4405


accuracy              f1         roc_auc         log_loss        
                 mean     std    mean     std    mean     std     mean     std
method                                                                        
Central        0.7253  0.0316  0.6031  0.0455  0.7775  0.0176   0.5236  0.0138
Client prior   0.6671  0.0406  0.5714  0.0673  0.6666  0.0146   0.5968  0.0223